## Query Compression - Domain Vocabulary Extraction
_Purpose: Summarize the query compression pipeline and goals._

The query compression pipeline:
* Builds a comprehensive domain vocabulary from fashion_with_brands.csv (brands, products, colors, sizes, materials)
* Extracts only domain-relevant tokens from customer queries
* Preserves semantic meaning while discarding irrelevant filler text
* Achieves higher precision than generic stop word removal

#### Imports

In [150]:
import pandas as pd
import re
import string
from collections import Counter
import json
from groq import Groq

#### Helper functions

In [151]:
# Pretty printing
def banner(title, pad = '----', width = 21):
    print(pad * width)
    print(title)
    print(pad * width)

# Remove punctuation and convert to lowercase
def sanitize_text(text):
    return str(text).lower().translate(str.maketrans('', '', string.punctuation))

#### Paths

In [152]:
query_csv_path = 'retrieval_evaluation_queries.csv'
fashion_csv_path = 'fashion_with_brands.csv'
output_queries_csv_path = 'compressed_retrieval_evaluation_queries.csv'
vocab_output_path = 'domain_vocabulary.json'

df = pd.read_csv(query_csv_path)
fashion_df = pd.read_csv(fashion_csv_path)

print(f"◉ Loaded {len(df)} queries, and {len(fashion_df)} products.")

◉ Loaded 10 queries, and 2906 products.


### Step 1: Build Domain Vocabulary from Fashion Dataset
_Purpose: Introduce vocabulary extraction from the catalog._

Extract all relevant domain terms from the product catalog to create a comprehensive vocabulary.

In [153]:
# Extract domain vocabulary from fashion_with_brands.csv
class DomainVocabularyBuilder:
    
    # Uses sets to automatically handle deduplication for each fashion category
    def __init__(self, fashion_df):
        self.fashion_df = fashion_df
        self.vocabulary = {
            'brands': set(), 'colors': set(), 'products': set(),
            'materials': set(), 'sizes': set(), 'styles': set(),
            'gender': set(), 'all_tokens': set()
        }
    
    # Scans CSV columns and build a dictionary of all the valid fashion terms it finds
    def extract_from_structured_columns(self):
        print("\n◉ Extracting from structured columns...")
        
        column_mapping = {
            'color': 'colors', 'size': 'sizes', 'material': 'materials',
            'style': 'styles', 'gender': 'gender'
        }
        
        for col in self.fashion_df.columns:
            col_lower = col.lower()
            for key, category in column_mapping.items():
                if key in col_lower:
                    tokens = {
                        token.strip().lower() 
                        for val in self.fashion_df[col].dropna().astype(str)
                        for token in val.split(',')
                        if len(token.strip().lower()) > 1
                    }
                    self.vocabulary[category].update(tokens)
                    print(f"    Extracted {len(tokens)} {category} from column '{col}'")
                    break
    
    # Extract brand names directly from the BrandName column
    def extract_brands(self):
        print("\n◉ Extracting brands from BrandName column...")
        
        brands = {
            brand.strip().lower() 
            for brand in self.fashion_df['BrandName'].dropna().astype(str)
            if len(brand.strip().lower()) > 1
        }
        
        self.vocabulary['brands'].update(brands)
        print(f"    Extracted {len(self.vocabulary['brands'])} brand names")
    
    # Extract product types directly from ProductType column
    def extract_products_from_type(self):
        print("\n◉ Extracting products from ProductType column...")
        
        products = {
            product.strip().lower()
            for product in self.fashion_df['ProductType'].dropna().astype(str)
            if len(product.strip().lower()) > 1
        }
        
        self.vocabulary['products'].update(products)
        print(f"    Extracted {len(products)} product types")
    
    # Add common fashion descriptors using LLM
    def add_common_fashion_descriptors(self, groq_client):
        print("\n◉ Adding common fashion descriptors using LLM...")
        
        # Add base gender terms
        gender_terms = {'men', 'mens', "men's", 'women', 'womens', "women's", 'unisex',
                       'boys', 'boy', "boy's", 'girls', 'girl', "girl's", 'kids', "kids'",
                       'infant', 'baby', 'toddler', 'children', 'kidswear', 'son', 'daughter'}
        self.vocabulary['gender'].update(gender_terms)
        print(f"    Added {len(gender_terms)} base gender terms")
        
        prompt = """Generate a comprehensive list of fashion-related terms in JSON format.
Include:
- gender (target audience terms like men, women, boys, girls, kids, unisex, etc.)
- styles (most known and common daily fashion styles)
- materials (fabrics and textures)
- colors (including modern/trendy color names)

Return ONLY a JSON object with these 4 keys, each containing an array of lowercase terms.
Example: {"gender": ["mens", "womens", "kids"], "colors": ["burgundy", "sage"], "styles": ["preppy", "grunge"], ...}"""
        
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=1200
        )
        
        llm_output = response.choices[0].message.content.strip()
        
        # Remove markdown code blocks if present
        if llm_output.startswith('```'):
            llm_output = llm_output.split('\n', 1)[1]
            llm_output = llm_output.rsplit('```', 1)[0]
        
        llm_vocab = json.loads(llm_output)
        
        # Add LLM-generated terms to vocabulary
        for category in ['gender', 'colors', 'styles', 'materials']:
            if category in llm_vocab:
                new_terms = set(llm_vocab[category])
                self.vocabulary[category].update(new_terms)
                print(f"    Added {len(new_terms)} {category} from LLM")
    
    # Build complete domain vocabulary
    def build_vocabulary(self, groq_client):
        banner("Building dynamic domain vocabulary from fashion_with_brands.csv")
        
        self.extract_from_structured_columns()
        self.extract_brands()
        self.extract_products_from_type()
        self.add_common_fashion_descriptors(groq_client)
        
        # Update all_tokens
        self.vocabulary['all_tokens'].clear()
        for category, tokens in self.vocabulary.items():
            if category != 'all_tokens' and isinstance(tokens, set):
                self.vocabulary['all_tokens'].update(tokens)
        
        banner("Domain Vocabulary Summary")
        print(f"\n◉ Total domain vocabulary: {len(self.vocabulary['all_tokens'])} unique tokens")
        for category, tokens in self.vocabulary.items():
            if category != 'all_tokens' and tokens:
                print(f"    {category:15s}: {len(tokens):4d} terms")
        
        return self.vocabulary

### Step 1.5: Augment Vocabulary with Groq LLM
_Purpose: Use LLM to generate additional fashion terms._

Enhance vocabulary with AI-generated synonyms and modern fashion terms.

In [ ]:
groq_client = Groq(api_key=" ")

In [155]:
# Create vocabulary builder and build vocabulary with LLM augmentation
vocab_builder = DomainVocabularyBuilder(fashion_df)
DOMAIN_VOCAB = vocab_builder.build_vocabulary(groq_client)

------------------------------------------------------------------------------------
Building dynamic domain vocabulary from fashion_with_brands.csv
------------------------------------------------------------------------------------

◉ Extracting from structured columns...
    Extracted 4 gender from column 'Gender'
    Extracted 7 sizes from column 'Size'

◉ Extracting brands from BrandName column...
    Extracted 84 brand names

◉ Extracting products from ProductType column...
    Extracted 31 product types

◉ Adding common fashion descriptors using LLM...
    Added 22 base gender terms
    Added 11 gender from LLM
    Added 77 colors from LLM
    Added 24 styles from LLM
    Added 25 materials from LLM
------------------------------------------------------------------------------------
Domain Vocabulary Summary
------------------------------------------------------------------------------------

◉ Total domain vocabulary: 271 unique tokens
    brands         :   84 terms
    colors

## Step 2: Smart Query Compression Function
_Purpose: Introduce the query compression step._

Extract only domain-relevant tokens from queries using the built vocabulary.

In [161]:
# Compress query by extracting only domain-relevant tokens
def smart_compress_query(query, domain_vocab):
    query = str(query).lower()
    
    # Remove typical greeting & polite/chatty prefixes
    greetings = r'^(hey|hi|hello|greetings)(\s+there)?\s+'
    conversational = [
        r'\b(i\s+)?am\s+looking\s+for\b', r'\blooking\s+for\b',
        r'\bdo\s+you\s+have\s+(any\s+)?\b', r'\bcan\s+you\s+show\s+(me\s+)?\b',
        r'\b(show\s+me|i\s+(want|need)|please|thanks?|thank\s+you)\b'
    ]
    query = re.sub(greetings, '', query)
    for pattern in conversational:
        query = re.sub(pattern, ' ', query)
    
    # Helper defined earlier
    query = sanitize_text(query)
    tokens = query.split()
    
    relevant_tokens = []
    structural_words = {'with', 'in', 'for', 'and', 'or', 'size', 'color', 'type', 'my', 'from'}
    possessive_words = {'my', 'your', 'his', 'her', 'their', 'our'}
    gender_terms = set(domain_vocab.get('gender', []))
    
    for i, token in enumerate(tokens):
        clean_token = token.strip()
        
        if clean_token in domain_vocab['all_tokens']:
            relevant_tokens.append(clean_token)
        elif clean_token in possessive_words:
            # keep "my" only if followed by "men"/"women"/"kids"…
            next_token = tokens[i + 1].strip() if i + 1 < len(tokens) else ''
            if next_token in gender_terms:
                relevant_tokens.append(clean_token)
        elif clean_token in structural_words and relevant_tokens:
            # keep connectors only if something was already kept
            relevant_tokens.append(clean_token)
        elif clean_token.isdigit():
            # keep sizes like 38 40 42, years, quantities…
            relevant_tokens.append(clean_token)
    
    # Final formatting
    compressed = ' '.join(relevant_tokens)
    return re.sub(r'\s+', ' ', compressed).strip()


# Apply smart compression to each query and store in new column + print sample results
df['smart_compressed'] = df['query'].apply(lambda q: smart_compress_query(q, DOMAIN_VOCAB))

banner('Compression Results (Sample)')
for idx, (_, row) in enumerate(df.head(10).iterrows(), 1):
    reduction = 100 * (1 - len(row['smart_compressed']) / len(row['query'])) if row['smart_compressed'] else 0
    orig = row['query'][:40] + "..." if len(row['query']) > 40 else row['query']
    print(f"\n{idx}. {orig}")
    print(f"   → {row['smart_compressed']} ({reduction:.0f}%)")

------------------------------------------------------------------------------------
Compression Results (Sample)
------------------------------------------------------------------------------------

1. Hey, do you have any white tops for girl...
   → white tops for girls from and in large size (44%)

2. I'm looking for a black top for my daugh...
   → black for my daughter (70%)

3. I'm looking for a blue casual top for my...
   → blue casual for my daughter from and (60%)

4. Hi, I'm looking for a cute pink top for ...
   → pink for my daughter with (73%)

5. I'm looking for some black capris for my...
   → black capris for my daughter (58%)

6. Hey, do you have any white tops for girl...
   → white tops for girls in medium size (41%)

7. Do you have any pink tops for girls in s...
   → pink tops for girls in size medium (33%)

8. Hi, I'm looking for a red top for my dau...
   → red for my daughter in size (69%)

9. I'm looking for some olive green capris ...
   → olive green capris 

## Step 3: Analyze and Save Results
_Purpose: Introduce summary statistics and saving outputs._

Compute statistics, analyze vocabulary coverage, and save compressed queries.

In [157]:
original_len = df['query'].str.len().sum()
compressed_len = df['smart_compressed'].str.len().sum()
reduction = 100 * (1 - compressed_len / original_len)
empty = (df['smart_compressed'] == '').sum()

banner(f'Compression Summary: {len(df)} queries')
print(f"Total compression: {reduction:.1f}%")
print(f"Original chars: {original_len:,} → Compressed: {compressed_len:,}")
if empty:
    print(f"⚠️ {empty} queries with no domain content")

------------------------------------------------------------------------------------
Compression Summary: 10 queries
------------------------------------------------------------------------------------
Total compression: 59.2%
Original chars: 762 → Compressed: 311


## Step 4: Analyze Vocabulary Coverage
_Purpose: Introduce coverage analysis of the vocabulary._

Check how well our domain vocabulary covers the query terms.

In [158]:
# Measures how well is the collected DOMAIN_VOCAB['all_tokens']

banner('Vocabulary Coverage')

total_tokens = covered_tokens = 0
uncovered = Counter()

for query in df['query']:
    query_clean = sanitize_text(query)
    for token in query_clean.split():
        if len(token) > 1:
            total_tokens += 1
            if token in DOMAIN_VOCAB['all_tokens']:
                covered_tokens += 1
            else:
                uncovered[token] += 1

coverage = 100 * covered_tokens / max(total_tokens, 1)
print(f"\nTotal tokens: {total_tokens} | Covered: {covered_tokens} ({coverage:.1f}%)")
print(f"\nTop 10 uncovered tokens:")
for token, count in uncovered.most_common(10):
    print(f"  {token}: {count}x")

------------------------------------------------------------------------------------
Vocabulary Coverage
------------------------------------------------------------------------------------

Total tokens: 147 | Covered: 31 (21.1%)

Top 10 uncovered tokens:
  for: 17x
  you: 11x
  have: 10x
  do: 9x
  im: 7x
  looking: 7x
  my: 7x
  any: 5x
  in: 4x
  size: 4x


Mesh a7san coverage, need more queries.

## Step 5: Compression Statistics
_Purpose: Introduce detailed compression metrics._

Calculate overall compression performance.

In [159]:
banner('Detailed Compression Statistics')

orig_len = df['query'].str.len()
comp_len = df['smart_compressed'].str.len()
print(f"\nOriginal: {orig_len.sum():,} chars (avg: {orig_len.mean():.0f}, min: {orig_len.min()}, max: {orig_len.max()})")
print(f"Compressed: {comp_len.sum():,} chars (avg: {comp_len.mean():.0f}, min: {comp_len.min()}, max: {comp_len.max()})")
print(f"\nAvg chars/query removed: {(orig_len - comp_len).mean():.0f}")

------------------------------------------------------------------------------------
Detailed Compression Statistics
------------------------------------------------------------------------------------

Original: 762 chars (avg: 76, min: 51, max: 92)
Compressed: 311 chars (avg: 31, min: 21, max: 43)

Avg chars/query removed: 45


## Step 6: Save Results
_Purpose: Introduce saving compressed outputs._

Save the compressed queries to the CSV file.

In [160]:
import os
import json

df_to_save = df[['query']].copy()
df_to_save['query'] = df['smart_compressed']
df_to_save.to_csv(output_queries_csv_path, index=False)

# Use the directory where the CSV files are located
# This should be the chatbot directory
csv_dir = os.path.dirname(os.path.abspath(query_csv_path)) or os.getcwd()
vocab_json_path = os.path.join(csv_dir, vocab_output_path)

# Build vocabulary as JSON structure
vocab_json = {category: sorted(list(DOMAIN_VOCAB.get(category, set()))) 
              for category in ['brands', 'colors', 'sizes', 'materials', 'styles', 'products', 'gender']}

# Write to JSON file
with open(vocab_json_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_json, f, indent=2, ensure_ascii=False)

banner('Results Saved')
print(f"✓ CSV: {output_queries_csv_path}")
print(f"✓ Vocabulary (JSON): {vocab_json_path}")
print(f"✓ Total rows: {len(df_to_save)}")
print(f"\nDataFrame preview:")
print(df_to_save.head().to_string())

------------------------------------------------------------------------------------
Results Saved
------------------------------------------------------------------------------------
✓ CSV: compressed_retrieval_evaluation_queries.csv
✓ Vocabulary (JSON): e:\OneDrive - Faculty of Computer and Information Sciences (Ain Shams University)\GP\SocaiLift\SociaLift\chatbot\domain_vocabulary.json
✓ Total rows: 10

DataFrame preview:
                                         query
0  white tops for girls from and in large size
1                        black for my daughter
2         blue casual for my daughter from and
3                    pink for my daughter with
4                 black capris for my daughter


## LangGraph Implementation
_Purpose: Graph-based orchestration of the query compression pipeline._

This section implements a LangGraph state machine with sequential nodes:
- **Load Data Node**: Load CSV files and initialize dataframes
- **Build Vocabulary Node**: Extract domain terms from fashion dataset
- **Augment Vocabulary Node**: Add LLM-generated fashion descriptors
- **Compress Queries Node**: Apply compression to all queries
- **Analyze Results Node**: Calculate statistics and coverage
- **Save Results Node**: Export compressed queries and vocabulary

In [ ]:
# LangGraph imports
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# Define the state schema for the compression pipeline
class CompressionState(TypedDict):
    """State schema for the query compression graph."""
    # Input paths
    query_csv_path: str
    fashion_csv_path: str
    output_queries_csv_path: str
    vocab_output_path: str
    
    # Data
    df: Optional[pd.DataFrame]
    fashion_df: Optional[pd.DataFrame]
    
    # Vocabulary
    domain_vocab: Optional[dict]
    groq_client: Optional[Groq]
    
    # Results
    compressed_df: Optional[pd.DataFrame]
    statistics: Optional[dict]
    
    # Status
    status: str

In [ ]:
# Node 1: Load Data
def load_data_node(state: CompressionState) -> CompressionState:
    """Load CSV files and initialize dataframes."""
    print("📂 Load Data Node: Loading CSV files...")
    
    df = pd.read_csv(state["query_csv_path"])
    fashion_df = pd.read_csv(state["fashion_csv_path"])
    
    state["df"] = df
    state["fashion_df"] = fashion_df
    state["status"] = "data_loaded"
    
    print(f"   ✓ Loaded {len(df)} queries and {len(fashion_df)} products")
    return state

# Node 2: Build Vocabulary
def build_vocabulary_node(state: CompressionState) -> CompressionState:
    """Extract domain vocabulary from fashion dataset."""
    print("\n🔤 Build Vocabulary Node: Extracting domain terms...")
    
    vocab_builder = DomainVocabularyBuilder(state["fashion_df"])
    
    # Extract from structured columns
    vocab_builder.extract_from_structured_columns()
    vocab_builder.extract_brands()
    vocab_builder.extract_products_from_type()
    
    state["domain_vocab"] = vocab_builder.vocabulary
    state["status"] = "vocabulary_built"
    
    print(f"   ✓ Extracted vocabulary with {len(vocab_builder.vocabulary['all_tokens'])} unique tokens")
    return state

# Node 3: Augment with LLM
def augment_vocabulary_node(state: CompressionState) -> CompressionState:
    """Augment vocabulary with LLM-generated fashion terms."""
    print("\n🤖 Augment Vocabulary Node: Adding LLM-generated terms...")
    
    vocab_builder = DomainVocabularyBuilder(state["fashion_df"])
    vocab_builder.vocabulary = state["domain_vocab"]
    
    # Add LLM-generated descriptors
    vocab_builder.add_common_fashion_descriptors(state["groq_client"])
    
    # Update all_tokens
    vocab_builder.vocabulary['all_tokens'].clear()
    for category, tokens in vocab_builder.vocabulary.items():
        if category != 'all_tokens' and isinstance(tokens, set):
            vocab_builder.vocabulary['all_tokens'].update(tokens)
    
    state["domain_vocab"] = vocab_builder.vocabulary
    state["status"] = "vocabulary_augmented"
    
    print(f"   ✓ Augmented vocabulary now has {len(vocab_builder.vocabulary['all_tokens'])} unique tokens")
    return state

# Node 4: Compress Queries
def compress_queries_node(state: CompressionState) -> CompressionState:
    """Apply compression to all queries."""
    print("\n🗜️ Compress Queries Node: Applying compression...")
    
    df = state["df"].copy()
    df['smart_compressed'] = df['query'].apply(
        lambda q: smart_compress_query(q, state["domain_vocab"])
    )
    
    state["compressed_df"] = df
    state["status"] = "queries_compressed"
    
    original_len = df['query'].str.len().sum()
    compressed_len = df['smart_compressed'].str.len().sum()
    reduction = 100 * (1 - compressed_len / original_len)
    
    print(f"   ✓ Compressed {len(df)} queries ({reduction:.1f}% reduction)")
    return state

# Node 5: Analyze Results
def analyze_results_node(state: CompressionState) -> CompressionState:
    """Calculate statistics and coverage metrics."""
    print("\n📊 Analyze Results Node: Computing statistics...")
    
    df = state["compressed_df"]
    domain_vocab = state["domain_vocab"]
    
    # Compression statistics
    original_len = df['query'].str.len().sum()
    compressed_len = df['smart_compressed'].str.len().sum()
    reduction = 100 * (1 - compressed_len / original_len)
    empty = (df['smart_compressed'] == '').sum()
    
    # Vocabulary coverage
    total_tokens = covered_tokens = 0
    uncovered = Counter()
    
    for query in df['query']:
        query_clean = sanitize_text(query)
        for token in query_clean.split():
            if len(token) > 1:
                total_tokens += 1
                if token in domain_vocab['all_tokens']:
                    covered_tokens += 1
                else:
                    uncovered[token] += 1
    
    coverage = 100 * covered_tokens / max(total_tokens, 1)
    
    state["statistics"] = {
        "total_queries": len(df),
        "original_chars": original_len,
        "compressed_chars": compressed_len,
        "compression_rate": reduction,
        "empty_queries": empty,
        "vocabulary_coverage": coverage,
        "total_tokens": total_tokens,
        "covered_tokens": covered_tokens,
        "top_uncovered": uncovered.most_common(10)
    }
    state["status"] = "results_analyzed"
    
    print(f"   ✓ Compression: {reduction:.1f}% | Coverage: {coverage:.1f}%")
    return state

# Node 6: Save Results
def save_results_node(state: CompressionState) -> CompressionState:
    """Export compressed queries and vocabulary to files."""
    print("\n💾 Save Results Node: Exporting files...")
    
    df = state["compressed_df"]
    domain_vocab = state["domain_vocab"]
    
    # Save compressed queries
    df_to_save = df[['query']].copy()
    df_to_save['query'] = df['smart_compressed']
    df_to_save.to_csv(state["output_queries_csv_path"], index=False)
    
    # Save vocabulary as JSON
    csv_dir = os.path.dirname(os.path.abspath(state["query_csv_path"])) or os.getcwd()
    vocab_json_path = os.path.join(csv_dir, state["vocab_output_path"])
    
    vocab_json = {
        category: sorted(list(domain_vocab.get(category, set())))
        for category in ['brands', 'colors', 'sizes', 'materials', 'styles', 'products', 'gender']
    }
    
    with open(vocab_json_path, 'w', encoding='utf-8') as f:
        json.dump(vocab_json, f, indent=2, ensure_ascii=False)
    
    state["status"] = "results_saved"
    
    print(f"   ✓ Saved to {state['output_queries_csv_path']}")
    print(f"   ✓ Saved to {vocab_json_path}")
    return state

In [ ]:
# Parent Node: Execute full compression pipeline
def compression_pipeline_node(state: CompressionState) -> CompressionState:
    """
    Parent node that executes all sub-nodes in the compression pipeline.
    This orchestrates the entire query compression workflow.
    """
    banner("Query Compression Pipeline - Parent Node", pad='====')
    
    # Execute each sub-node in sequence
    state = load_data_node(state)
    state = build_vocabulary_node(state)
    state = augment_vocabulary_node(state)
    state = compress_queries_node(state)
    state = analyze_results_node(state)
    state = save_results_node(state)
    
    print("\n" + "=" * 80)
    print("✅ Compression Pipeline Parent Node: All sub-nodes completed!")
    print("=" * 80)
    
    return state

# Build the query compression graph with parent node
def create_compression_graph() -> StateGraph:
    """Create and compile the LangGraph with a parent compression node."""
    
    # Initialize the graph
    workflow = StateGraph(CompressionState)
    
    # Add the parent node that orchestrates all sub-nodes
    workflow.add_node("compression_pipeline", compression_pipeline_node)
    
    # Set entry point to parent node
    workflow.set_entry_point("compression_pipeline")
    
    # Parent node goes directly to END
    workflow.add_edge("compression_pipeline", END)
    
    # Compile the graph
    graph = workflow.compile()
    print("✅ Query compression graph compiled successfully!")
    print("   Architecture: Entry → compression_pipeline (parent node) → END")
    return graph

# Create the graph
compression_graph = create_compression_graph()

In [ ]:
# Execute the full query compression pipeline using LangGraph
banner("LangGraph Pipeline Execution", pad='====')

# Initialize state with configuration
initial_state = {
    "query_csv_path": 'retrieval_evaluation_queries.csv',
    "fashion_csv_path": 'fashion_with_brands.csv',
    "output_queries_csv_path": 'compressed_retrieval_evaluation_queries.csv',
    "vocab_output_path": 'domain_vocabulary.json',
    "df": None,
    "fashion_df": None,
    "domain_vocab": None,
    "groq_client": groq_client,  # Use the previously initialized Groq client
    "compressed_df": None,
    "statistics": None,
    "status": "initialized"
}

# Execute the graph
final_state = compression_graph.invoke(initial_state)

# Display final statistics
banner("Pipeline Completion Summary", pad='====')
stats = final_state['statistics']
print(f"\n📊 Compression Statistics:")
print(f"   Total Queries: {stats['total_queries']}")
print(f"   Compression Rate: {stats['compression_rate']:.1f}%")
print(f"   Original Chars: {stats['original_chars']:,}")
print(f"   Compressed Chars: {stats['compressed_chars']:,}")
print(f"   Empty Queries: {stats['empty_queries']}")
print(f"\n📚 Vocabulary Coverage:")
print(f"   Total Tokens: {stats['total_tokens']}")
print(f"   Covered Tokens: {stats['covered_tokens']}")
print(f"   Coverage Rate: {stats['vocabulary_coverage']:.1f}%")
print(f"\n🔝 Top 10 Uncovered Tokens:")
for token, count in stats['top_uncovered']:
    print(f"   {token}: {count}x")

print(f"\n✅ Final Status: {final_state['status']}")
print("=" * 80)

In [ ]:
# Display sample compression results from the graph execution
banner("Sample Compression Results", pad='====')

compressed_df = final_state['compressed_df']

print("\n📝 First 10 Query Compressions:")
for idx, (_, row) in enumerate(compressed_df.head(10).iterrows(), 1):
    reduction = 100 * (1 - len(row['smart_compressed']) / len(row['query'])) if row['smart_compressed'] else 0
    orig = row['query'][:50] + "..." if len(row['query']) > 50 else row['query']
    comp = row['smart_compressed'][:50] + "..." if len(row['smart_compressed']) > 50 else row['smart_compressed']
    
    print(f"\n{idx}. Original: {orig}")
    print(f"   Compressed: {comp}")
    print(f"   Reduction: {reduction:.1f}%")

print("\n" + "=" * 80)